# AnyLoc retrieval for dataset-3 localization

This notebook samples the two ARKit recordings, runs AnyLoc retrieval from query frames to reference frames, streams the similarity matrix to CSV, and exports the top matches to Excel. It does **not** estimate 6-DoF poses; `hloc.ipynb` performs that stage.

## Configuration

The default stride keeps every fifth 10 Hz frame, so the experiment runs at about 2 FPS (158 reference images and 155 queries).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'indoor_vpr').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not find the indoor-VPR project root.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from indoor_vpr import DatasetConfig, ImageDataset, create_algorithm, stream_vpr_similarity_csv
from indoor_vpr.localization import prepare_localization_frames, export_anyloc_retrieval_xlsx

DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'dataset-3'
REFERENCE_RECORDING = DATASET_ROOT / 'SR_2026-08-30_01-46-38'
QUERY_RECORDING = DATASET_ROOT / 'SR_2026-08-30_01-48-04'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'localization'
IMAGE_ROOT = OUTPUT_DIR / 'images'

FRAME_STRIDE = 5
OVERWRITE_EXTRACTED_FRAMES = False
TOP_K = 20
SIMILARITY_CSV = OUTPUT_DIR / 'anyloc_similarity.csv'
RETRIEVAL_XLSX = OUTPUT_DIR / 'anyloc_retrieval.xlsx'

ANYLOC_OPTIONS = {
    'model_name': 'dinov2_vitg14',
    'layer': 31,
    'facet': 'value',
    'num_clusters': 32,
    'vocabulary_path': None,
    'max_image_size': 1024,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Reference:', REFERENCE_RECORDING.name)
print('Query:    ', QUERY_RECORDING.name)
print('Output:   ', OUTPUT_DIR)


Reference: SR_2026-08-30_01-46-38
Query:     SR_2026-08-30_01-48-04
Output:    /Users/armin/Documents/indoor-VPR/outputs/localization


## Extract synchronized frames

Each JPEG keeps the corresponding ARKit pose and intrinsics in a manifest. Existing complete extractions are reused.

In [2]:
REFERENCE_MANIFEST, QUERY_MANIFEST = prepare_localization_frames(
    REFERENCE_RECORDING,
    QUERY_RECORDING,
    IMAGE_ROOT,
    frame_stride=FRAME_STRIDE,
    overwrite=OVERWRITE_EXTRACTED_FRAMES,
)
print('Reference manifest:', REFERENCE_MANIFEST)
print('Query manifest:    ', QUERY_MANIFEST)


Reference manifest: /Users/armin/Documents/indoor-VPR/outputs/localization/images/reference_manifest.csv
Query manifest:     /Users/armin/Documents/indoor-VPR/outputs/localization/images/query_manifest.csv


## Run AnyLoc retrieval

The database descriptors stay in memory, but similarity rows are written incrementally to CSV.

In [3]:
dataset = ImageDataset.from_config(
    DatasetConfig(
        database_dir=IMAGE_ROOT / 'reference',
        query_dir=IMAGE_ROOT / 'query',
    )
)
print(dataset.summary())

algorithm = create_algorithm('anyloc', **ANYLOC_OPTIONS)
results = stream_vpr_similarity_csv(
    dataset,
    algorithm,
    csv_path=SIMILARITY_CSV,
    query_batch_size=1,
    progress_every=25,
    top_k=TOP_K,
)
print('Similarity CSV:', SIMILARITY_CSV)


Database: 158 images | Queries: 155 images


Using cache found in /Users/armin/.cache/torch/hub/facebookresearch_dinov2_main
/Users/armin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/armin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/armin/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Encoding database descriptors...
Encoded 158/158 database images.
Encoding queries and streaming similarity rows...
Wrote 25/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 50/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 75/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 100/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 125/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 150/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Wrote 155/155 query rows to /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv
Similarity CSV: /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_similarity.csv


## Export top matches to Excel

This reads the CSV one row at a time, so it does not recreate the complete similarity matrix in memory.

In [4]:
export_anyloc_retrieval_xlsx(
    SIMILARITY_CSV,
    REFERENCE_MANIFEST,
    QUERY_MANIFEST,
    RETRIEVAL_XLSX,
    top_k=TOP_K,
)
print('AnyLoc retrieval workbook:', RETRIEVAL_XLSX)


AnyLoc retrieval workbook: /Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_retrieval.xlsx


## Re-export without rerunning AnyLoc

After the expensive retrieval has completed once, run only this cell to change the Excel top-k.

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'indoor_vpr').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not find the indoor-VPR project root.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from indoor_vpr.localization import export_anyloc_retrieval_xlsx

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'localization'
TOP_K_FOR_EXCEL = 20
export_anyloc_retrieval_xlsx(
    OUTPUT_DIR / 'anyloc_similarity.csv',
    OUTPUT_DIR / 'images' / 'reference_manifest.csv',
    OUTPUT_DIR / 'images' / 'query_manifest.csv',
    OUTPUT_DIR / 'anyloc_retrieval.xlsx',
    top_k=TOP_K_FOR_EXCEL,
)
print(OUTPUT_DIR / 'anyloc_retrieval.xlsx')


/Users/armin/Documents/indoor-VPR/outputs/localization/anyloc_retrieval.xlsx
